read data from bronze layer

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.driver"
silver_table = f"{catalog_name}.{silver_schema}.driver"

In [0]:
driver_df = spark.table(bronze_table)
display(driver_df)

keep only column require for analytics drop url column

In [0]:
from pyspark.sql import functions as F

selected_driver_df = driver_df.select(
    F.col("driverId"),
    F.col("name"),
    F.col("dateOfBirth"),
    F.col("nationality"),
    F.col("ingestion_date"),
    F.col("source_file")
)

standardise column with snaka case(countryName ==> country_name, data ---> race_data)

In [0]:
standardise_driver_df = selected_driver_df.withColumnsRenamed({
                                    "driverId":"driver_id",
                                    "dateOfBirth":"date_of_birth",
                                })

#### Drop Duplicate Record

In [0]:
print(standardise_driver_df.count())

In [0]:
driver_distint_df = standardise_driver_df.dropDuplicates(["driver_id"]) # driver_id is a primary key

In [0]:
print(driver_distint_df.count())

##### create driver_name from name(givename and family name)

In [0]:
concatened_driver_df = driver_distint_df.withColumn("driver_name",F.initcap(F.concat_ws(" ",F.col("name.givenName"),F.col("name.FamilyName")))).drop(F.col("name"))



#### transform values of column nationality to title case

In [0]:
driver_final_df = concatened_driver_df.withColumn("nationality",F.initcap(F.col("nationality")))
# display(races_final_df)

#### write data to silver layer

In [0]:
driver_final_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))